# 02 — Análisis exploratorio y OE1 (estacionariedad)

Cubre el **Objetivo Específico 1**: caracterizar las propiedades estadísticas de las series
(distribución, volatilidad, estacionariedad, orden de integración).

La conclusión I(0) vs I(1) de este notebook **decide la estrategia de los OE2–OE4**.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config as C
from src import stationarity as st

sns.set_theme(style='whitegrid')

In [ ]:
# Cargar lo construido en el notebook 01
retornos_m = pd.read_parquet(C.DATA_INTERIM / 'retornos_mensuales.parquet')
# macro_global, macro_local: unir aquí una vez descargados
retornos_m.describe().T

## 1. Estadística descriptiva y distribución de retornos
Esperado en retornos financieros: media ~0, exceso de curtosis (colas pesadas), posible asimetría.

In [ ]:
from scipy import stats
desc = pd.DataFrame({
    'media': retornos_m.mean(),
    'desv': retornos_m.std(),
    'asimetria': retornos_m.skew(),
    'curtosis': retornos_m.kurt(),
    'min': retornos_m.min(),
    'max': retornos_m.max(),
})
desc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
retornos_m.plot(ax=axes[0], title='Retornos mensuales por empresa', alpha=.8)
retornos_m.plot.hist(ax=axes[1], bins=40, alpha=.5, title='Distribución de retornos')
plt.tight_layout(); plt.savefig(C.FIGURES / 'eda_retornos.png', dpi=120)

## 2. Correlaciones (mapa de calor)
Útil para detectar multicolinealidad entre regresores macro antes de modelar.

In [ ]:
# Reemplazar 'datos' por el DataFrame que combine retornos + macro ya alineados
datos = retornos_m.copy()
plt.figure(figsize=(8, 6))
sns.heatmap(datos.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación'); plt.tight_layout()
plt.savefig(C.FIGURES / 'eda_correlaciones.png', dpi=120)

## 3. OE1 — Batería de estacionariedad / orden de integración
ADF + Phillips-Perron + KPSS + Zivot-Andrews. Conclusión robusta = ADF/PP y KPSS coinciden.

**Recordatorio de signo:** retornos -> I(0); precios/niveles (cobre, TC) -> típicamente I(1).
Aplica esta tabla TANTO a retornos (modelo OE2) COMO a los niveles (cobre, TC, acción)
que entrarán a cointegración (OE3).

In [ ]:
tabla_oe1 = st.tabla_estacionariedad(retornos_m.dropna())
tabla_oe1

In [ ]:
# Exportar la tabla para el Capítulo 4
tabla_oe1.to_csv(C.TABLES / 'oe1_estacionariedad.csv')
tabla_oe1.to_latex(C.TABLES / 'oe1_estacionariedad.tex', float_format='%.3f')
print('Tabla OE1 exportada a', C.TABLES)

## 4. Lectura de resultados (OE1)
Documenta aquí, en prosa, qué resultó I(0) y qué I(1). Esa clasificación define:
- Variables I(0) -> entran en niveles/retornos a los modelos de impacto (OE2).
- Variables I(1) -> se diferencian para OE2, y en **nivel** van a cointegración (OE3).

Próximo notebook: `03_oe2_panel.ipynb` (modelo multifactorial / panel FE).